In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
data_path = r"C:\Users\JINESHA GANDHI\Downloads\PetImages\PetImages"

# Data preprocessing
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    data_path,
    target_size=(64,64),
    batch_size=32,
    class_mode='sparse',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_path,
    target_size=(64,64),
    batch_size=32,
    class_mode='sparse',
    subset='validation'
)

num_classes = len(train_data.class_indices)
class_names = list(train_data.class_indices.keys())

Found 20000 images belonging to 2 classes.
Found 4998 images belonging to 2 classes.


In [3]:
# VGG16
base_vgg = tf.keras.applications.VGG16(
    input_shape=(64,64,3),
    include_top=False,
    weights='imagenet'
)

base_vgg.trainable = False

model_vgg = models.Sequential([
    base_vgg,
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_vgg.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

history_vgg = model_vgg.fit(train_data, validation_data=val_data, epochs=3)


Epoch 1/3
358/625 ━━━━━━━━━━━━━━━━━━━━ 54s 203ms/step - accuracy: 0.7431 - loss: 0.5160

C:\Users\JINESHA GANDHI\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


625/625 ━━━━━━━━━━━━━━━━━━━━ 158s 251ms/step - accuracy: 0.7850 - loss: 0.4517 - val_accuracy: 0.8111 - val_loss: 0.4111
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 188s 301ms/step - accuracy: 0.8128 - loss: 0.4026 - val_accuracy: 0.8065 - val_loss: 0.4110
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 192s 307ms/step - accuracy: 0.8209 - loss: 0.3826 - val_accuracy: 0.8113 - val_loss: 0.3974


In [ ]:
#ResNet50
base_resnet = tf.keras.applications.ResNet50(
    input_shape=(64,64,3),
    include_top=False,
    weights='imagenet'
)

base_resnet.trainable = False

model_resnet = models.Sequential([
    base_resnet,
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model_resnet.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

history_resnet = model_resnet.fit(train_data, validation_data=val_data, epochs=3)

Epoch 1/3
420/625 ━━━━━━━━━━━━━━━━━━━━ 32s 159ms/step - accuracy: 0.5639 - loss: 0.6866

In [ ]:
#COMPARISON
print("VGG Accuracy:", history_vgg.history['val_accuracy'][-1])
print("ResNet Accuracy:", history_resnet.history['val_accuracy'][-1])

plt.plot(history_vgg.history['val_accuracy'], label='VGG')
plt.plot(history_resnet.history['val_accuracy'], label='ResNet')
plt.title("Model Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()